# Understanding Monetary Policy Movements: The Chilean Case

In [ ]:
import numpy as np
import plotly.graph_objects as go
import pandas as pd
from scipy.stats import *
from statsmodels.tsa.stattools import adfuller, kpss

In [2]:
panel_df = pd.read_excel('macrodata.xlsx',sheet_name='panel_data')
policy_df = pd.read_excel('macrodata.xlsx',sheet_name='policy_rate')

In [16]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=policy_df.date,y=policy_df.tpm, name='Policy Rate'))
fig.add_trace(go.Scatter(x=panel_df.meeting_date,y=panel_df.tpm, mode='markers', name='Meeting Date'))
fig.update_layout(
    title='Chilean Monetary Policy Rate Over Time',
    xaxis_title='Date',
    yaxis_title='Policy Rate (%)',
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.2,
        xanchor="center",
        x=0.5
    )
)
fig.show()

# Annex 1: Definitions

## Unit Root

A time series is said to have a *unit root* if it is non-stationary and its current value depends on its previous value plus a stochastic error term. This implies that shocks to the series have *permanent effects*, meaning the series does not revert to a long-term mean or trend after being disturbed. A simple representation of a time series with a unit root is:

$$
y_t = y_{t-1} + \varepsilon_t
$$

where $ y_t $ is the value of the series at time $ t $, $ y_{t-1} $ is the value at time $ t-1 $, and $ \varepsilon_t $ is a white noise error term with mean zero and constant variance.  

A unit root process is typically associated with *stochastic trends*, making standard regression methods unreliable unless the series is transformed (e.g., by differencing). Many economic and financial time series exhibit unit roots, such as GDP levels, price indices, and exchange rates.

## Stationary Process

A *stationary* time series is one whose *statistical* properties remain constant over time. Formally, a time series $ \{y_t\} $ is weakly stationary (or covariance stationary) if:

- $ \mathbb{E}[y_t] = \mu $ (the mean is constant over time),
- $ \mathrm{Var}(y_t) = \sigma^2 $ (the variance is constant over time),
- $ \mathrm{Cov}(y_t, y_{t-k}) = \gamma_k $ (the autocovariance depends only on the lag $ k $, not on $ t $).

Stationarity is a key assumption in many time series methods (e.g., ARMA/ARIMA models, VAR models), because it ensures that model parameters are stable and forecasts are meaningful. Non-stationary data often need to be *differenced*, detrended, or transformed to achieve stationarity.

## Trend-Stationary Process

A *trend-stationary process* is a time series that is non-stationary in levels but can be rendered stationary by *removing a deterministic trend*. It is typically expressed as:

$$
y_t = \mu_t + \varepsilon_t
$$

where $ \mu_t $ represents a deterministic trend component (e.g., linear, quadratic, or higher-order polynomial) and $ \varepsilon_t $ is a stationary error term.  

Unlike a unit root process, shocks to a trend-stationary series are *temporary*---the series reverts to its deterministic trend path after disturbances. Detrending (e.g., by regression) is sufficient to obtain a stationary residual series.

## Augmented Dickey-Fuller (ADF) Test

The *Augmented Dickey-Fuller (ADF) test* is a statistical test commonly used to detect the presence of a unit root in a time series. The test extends the original Dickey-Fuller test by allowing for higher-order autoregressive processes, which improves robustness to serial correlation.

$$
\text{Hypotheses:} \quad 
\begin{cases}
H_0: \text{The time series has a unit root (non-stationary)} \\
H_1: \text{The time series is stationary}
\end{cases}
$$

A low p-value (typically $ p < 0.05 $) leads to rejection of $ H_0 $, indicating that the series is stationary. The ADF regression is typically specified as:

$$
\Delta y_t = \alpha + \beta t + \gamma y_{t-1} + \sum_{i=1}^{p} \delta_i \Delta y_{t-i} + \varepsilon_t
$$

where:
- $ \Delta y_t = y_t - y_{t-1} $ is the first difference,
- $ \alpha $ is a constant,
- $ \beta t $ allows for a deterministic trend,
- $ p $ is the number of lagged difference terms (selected to ensure white noise errors).

## Kwiatkowski–Phillips–Schmidt–Shin (KPSS) Test

The *KPSS test* provides a complementary approach to the ADF test by reversing the null hypothesis. It tests whether a series is stationary around a deterministic trend.

$$
\text{Hypotheses:} \quad 
\begin{cases}
H_0: \text{The time series is trend-stationary} \\
H_1: \text{The time series has a unit root (non-stationary)}
\end{cases}
$$

The KPSS test is particularly useful when used in conjunction with the ADF test:

- If ADF rejects and KPSS does not: evidence of stationarity.
- If ADF does not reject and KPSS rejects: evidence of non-stationarity.
- If both reject or both do not reject: the results may be inconclusive and further investigation is needed.


## p-value

The *p-value* is a fundamental concept in statistical hypothesis testing. It represents the probability of obtaining a test statistic at least as extreme as the one observed, assuming the null hypothesis $ H_0 $ is true. Formally:

$$
p\text{-value} = P(\text{Test Statistic} \geq \text{Observed Value} \mid H_0 \ \text{is true})
$$

A *low p-value* indicates that the observed data are unlikely under $ H_0 $, providing evidence against the null hypothesis. A *high p-value* suggests that the data are consistent with $ H_0 $.

In practice, a *significance level* ($ \alpha $) is chosen, commonly $ 0.05 $ or $ 0.01 $. The decision rule is:

$$
\text{Decision Rule:} \quad
\begin{cases}
\text{Reject } H_0 & \text{if } p\text{-value} < \alpha \\
\text{Fail to reject } H_0 & \text{if } p\text{-value} \geq \alpha
\end{cases}
$$

In the context of stationarity tests:

- For the ADF test, a low p-value indicates rejection of the null hypothesis of a unit root, implying the series is stationary.
- For the KPSS test, a low p-value indicates rejection of the null hypothesis of stationarity, implying the series is non-stationary.




## ADF Test

In [ ]:
result = adfuller(panel_df.tpm.dropna())
print('ADF Statistic:', result[0])
print('p-value:', result[1])
print('Number of Lags Used:', result[2])
print('Null Hypothesis Rejected:', result[1] < 0.05)
for key, value in result[4].items():
    print('Critical Value (%s): %.3f' % (key, value))

ADF Statistic: -4.065961907504031
p-value: 0.001101892947467032
Number of Lags Used: 4
Null Hypothesis Rejected (is stationary): True
Critical Value (1%): -3.452
Critical Value (5%): -2.871
Critical Value (10%): -2.572


## Kwiatkowski–Phillips–Schmidt–Shin (KPSS)

In [32]:
results = kpss(panel_df.tpm.dropna(), regression='c')
print('\nKPSS Statistic:', results[0])
print('p-value:', results[1])
print('Number of Lags Used:', results[2])
print('Null Hypothesis Rejected:', results[1] < 0.05)
for key, value in results[3].items():
    print('Critical Value (%s): %.3f' % (key, value))


KPSS Statistic: 0.20868575149601545
p-value: 0.1
Number of Lags Used: 10
Null Hypothesis Rejected: False
Critical Value (10%): 0.347
Critical Value (5%): 0.463
Critical Value (2.5%): 0.574
Critical Value (1%): 0.739


/var/folders/cp/l432p0ns1t38qmnyr_3l3r000000gn/T/ipykernel_66891/3902060053.py:1: InterpolationWarning:

The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.


